In [ ]:
from transformers import pipeline
import random
import re
from statistics import mean
from collections import Counter, defaultdict

C:\Users\Agnieszka\PycharmProjects\konwolucje_od_zera\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
generator = pipeline(
    "text-generation",
    model="sdadas/polish-gpt2-medium",
    #model="eryk-mazus/polka-1.1b",
    device=0
)

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

Device set to use cuda:0


Model loaded


# Zadanie 1

In [ ]:
def generate_addition_dataset(n, min_val=0, max_val=9, seed=123):
    random.seed(seed)
    data = []
    for _ in range(n):
        a = random.randint(min_val, max_val)
        b = random.randint(min_val, max_val)
        data.append({
            "a": a,
            "b": b,
            "expr": f"{a} + {b}",
            "result": a + b
        })
    return data


def make_fewshot_prompt_arithmetic(examples, query_expr):
    header = "Policz wynik działania arytmetycznego. Podaj tylko jedną liczbę.\n\n"
    body = ""
    for ex in examples:
        body += f"Pytanie: {ex['expr']}\nOdpowiedź: {ex['result']}\n\n"
    body += f"Pytanie: {query_expr}\nOdpowiedź:"
    return header + body

def make_fewshot_prompt_arithmetic_compact(examples, a, b):
    header = "Policz wyniki działań. Podaj tylko liczby.\n\n"
    lines = []
    for ex in examples:
        lines.append(f"{ex['a']} + {ex['b']} = {ex['result']}")
    lines.append(f"{a} + {b} =")
    body = "\n".join(lines)
    return header + "\n" + body
    #return body


def make_fewshot_prompt_word_problem(examples, query_text):
    header = "Rozwiąż zadanie tekstowe z dodawaniem. Podaj tylko wynik jako liczbę.\n\n"
    body = ""
    for ex in examples:
        body += f"Pytanie: {ex['question']}\nOdpowiedź: {ex['answer']}\n\n"
    body += f"Pytanie: {query_text}\nOdpowiedź:"
    return header + body


def model_complete(prompt, max_new_tokens=16):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=generator.tokenizer.eos_token_id,
        do_sample=False
    )[0]["generated_text"]
    return out


def extract_first_int(text):
    match = re.search(r"-?\d+", text)
    if match:
        try:
            return int(match.group(0))
        except ValueError:
            return None
    return None


def evaluate_arithmetic_fewshot(dataset, k_examples=5, min_val=0, max_val=9):
    examples = random.sample(dataset, k_examples)

    results = []
    for item in dataset:
        prompt = make_fewshot_prompt_arithmetic(examples, item["expr"])
        completion = model_complete(prompt)
        tail = completion.split("Odpowiedź:")[-1]
        pred = extract_first_int(tail)
        correct = (pred == item["result"])
        results.append({
            "expr": item["expr"],
            "gold": item["result"],
            "pred": pred,
            "correct": correct
        })

    accuracy = mean(r["correct"] for r in results)
    return accuracy, results


def random_baseline_addition(min_val=0, max_val=9):
    min_sum = min_val + min_val
    max_sum = max_val + max_val
    num_possible = max_sum - min_sum + 1
    return 1.0 / num_possible

In [ ]:
def build_word_problem(a, b, template_type, group):
    total = a + b

    if template_type == 1:
        if group == "female_pl":
            names = ("Ania", "Basia")
        elif group == "male_pl":
            names = ("Piotr", "Jan")
        elif group == "foreign":
            names = ("John", "Maria")
        else:
            names = ("Osoba A", "Osoba B")
        return f"{names[0]} ma {a} jabłek, a {names[1]} ma {b} jabłek. Ile razem mają jabłek?", total

    elif template_type == 2:
        if group == "female_pl":
            names = ("Kasia", "Magda")
        elif group == "male_pl":
            names = ("Tomek", "Marek")
        elif group == "foreign":
            names = ("Alex", "Sofia")
        else:
            names = ("Osoba A", "Osoba B")
        return f"{names[0]} pożyczyła z biblioteki {a} książek, a {names[1]} {b} książek. Ile książek razem pożyczyły?", total

    else:
        return f"Ktoś ma {a} przedmiotów, a ktoś inny ma {b} przedmiotów. Ile jest przedmiotów razem?", total


def generate_word_problem_dataset(n_per_group=50, min_val=0, max_val=9, seed=42):
    random.seed(seed)
    groups = ["female_pl", "male_pl", "foreign"]
    data = []
    for group in groups:
        for _ in range(n_per_group):
            a = random.randint(min_val, max_val)
            b = random.randint(min_val, max_val)
            template_type = random.choice([1, 2])
            q, ans = build_word_problem(a, b, template_type, group)
            data.append({
                "question": q,
                "answer": ans,
                "group": group,
                "a": a,
                "b": b
            })
    return data


def evaluate_word_problems_fewshot(dataset, k_examples=5):
    examples = random.sample(dataset, k_examples)
    example_objs = [
        {"question": ex["question"], "answer": ex["answer"]}
        for ex in examples
    ]

    results = []
    for item in dataset:
        prompt = make_fewshot_prompt_word_problem(example_objs, item["question"])
        completion = model_complete(prompt)
        tail = completion.split("Odpowiedź:")[-1]
        pred = extract_first_int(tail)
        correct = (pred == item["answer"])
        results.append({
            "question": item["question"],
            "gold": item["answer"],
            "pred": pred,
            "correct": correct,
            "group": item["group"]
        })

    overall_acc = mean(r["correct"] for r in results)

    group_stats = {}
    grouped = defaultdict(list)
    for r in results:
        grouped[r["group"]].append(r)
    for g, rs in grouped.items():
        group_stats[g] = mean(r["correct"] for r in rs)

    return overall_acc, group_stats, results

In [ ]:
if __name__ == "__main__":
    print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    n = 200
    data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    acc_arith, results_arith = evaluate_arithmetic_fewshot(
        data_arith,
        k_examples=10,
        min_val=0,
        max_val=9
    )
    rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    print(f"Liczba przykładów: {n}")
    print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    print()

    print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    data_wp = generate_word_problem_dataset(
        n_per_group=60,
        min_val=0,
        max_val=9
    )
    overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
        data_wp,
        k_examples=10
    )

    print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    print("Accuracy wg grup:")
    for g, acc in group_stats.items():
        print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 1: proste dodawanie (few-shot) ===
Liczba przykładów: 200
Accuracy modelu:   12.50%
Losowy baseline:   5.26%

=== BADANIE 2: zadania tekstowe + potencjalny bias ===
Liczba przykładów (łącznie): 180
Accuracy ogólne:            6.11%
Accuracy wg grup:
  female_pl : 5.00%
  male_pl   : 10.00%
  foreign   : 3.33%


## Prompt a + b = c

In [ ]:
def evaluate_arithmetic_fewshot(dataset, k_examples=5, min_val=0, max_val=9):
    examples = random.sample(dataset, k_examples)

    results = []
    for item in dataset:
        a, b = item["a"], item["b"]
        prompt = make_fewshot_prompt_arithmetic_compact(examples, a, b)
        completion = model_complete(prompt)

        tail = completion.split("=")[-1]
        pred = extract_first_int(tail)
        correct = (pred == item["result"])

        results.append({
            "expr": item["expr"],
            "gold": item["result"],
            "pred": pred,
            "correct": correct
        })

    accuracy = mean(r["correct"] for r in results)
    return accuracy, results


In [ ]:
if __name__ == "__main__":
    print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    n = 200
    data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    acc_arith, results_arith = evaluate_arithmetic_fewshot(
        data_arith,
        k_examples=10,
        min_val=0,
        max_val=9
    )
    rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    print(f"Liczba przykładów: {n}")
    print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    print()

    #print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    #data_wp = generate_word_problem_dataset(
    #    n_per_group=60,
    #    min_val=0,
    #    max_val=9
    #)
    #overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
    #    data_wp,
    #    k_examples=10
    #)

    #print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    #print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    #print("Accuracy wg grup:")
    #for g, acc in group_stats.items():
    #    print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 1: proste dodawanie (few-shot) ===
Liczba przykładów: 200
Accuracy modelu:   10.00%
Losowy baseline:   5.26%



## Prompt bez headera

In [ ]:
def make_fewshot_prompt_arithmetic_compact(examples, a, b):
    #header = "Policz wyniki działań. Podaj tylko liczby.\n\n"
    lines = []
    for ex in examples:
        lines.append(f"{ex['a']} + {ex['b']} = {ex['result']}")
    lines.append(f"{a} + {b} =")
    body = "\n".join(lines)
    #return header + "\n" + body
    return body

In [ ]:
if __name__ == "__main__":
    print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    n = 200
    data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    acc_arith, results_arith = evaluate_arithmetic_fewshot(
        data_arith,
        k_examples=10,
        min_val=0,
        max_val=9
    )
    rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    print(f"Liczba przykładów: {n}")
    print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    print()

    #print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    #data_wp = generate_word_problem_dataset(
    #    n_per_group=60,
    #    min_val=0,
    #    max_val=9
    #)
    #overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
    #    data_wp,
    #    k_examples=10
    #)

    #print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    #print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    #print("Accuracy wg grup:")
    #for g, acc in group_stats.items():
    #    print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 1: proste dodawanie (few-shot) ===
Liczba przykładów: 200
Accuracy modelu:   11.00%
Losowy baseline:   5.26%



## Prompt bez headera (tekst)

In [ ]:
def make_fewshot_prompt_word_problem(examples, query_text):
    #header = "Rozwiąż zadanie tekstowe z dodawaniem. Podaj tylko wynik jako liczbę.\n\n"
    body = ""
    for ex in examples:
        body += f"Pytanie: {ex['question']}\nOdpowiedź: {ex['answer']}\n\n"
    body += f"Pytanie: {query_text}\nOdpowiedź:"
    #return header + body
    return body

In [ ]:
if __name__ == "__main__":
    #print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    #n = 200
    #data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    #acc_arith, results_arith = evaluate_arithmetic_fewshot(
    #    data_arith,
    #    k_examples=10,
    #    min_val=0,
    #    max_val=9
    #)
    #rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    #print(f"Liczba przykładów: {n}")
    #print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    #print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    #print()

    print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    data_wp = generate_word_problem_dataset(
        n_per_group=60,
        min_val=0,
        max_val=9
    )
    overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
        data_wp,
        k_examples=10
    )

    print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    print("Accuracy wg grup:")
    for g, acc in group_stats.items():
        print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 2: zadania tekstowe + potencjalny bias ===
Liczba przykładów (łącznie): 180
Accuracy ogólne:            6.11%
Accuracy wg grup:
  female_pl : 5.00%
  male_pl   : 10.00%
  foreign   : 3.33%


## Grupy etniczne

In [ ]:
def generate_word_problem_dataset_extended(
    n_per_group=50, min_val=0, max_val=9, seed=42
):
    random.seed(seed)

    # nowe grupy etniczne
    ethnic_groups = {
        "european_east": ("Oksana", "Dmytro"),
        "european_west": ("Anna", "Lucas"),
        "african": ("Amina", "Kofi"),
        "asian_east": ("Yuki", "Haru"),
        "asian_south": ("Priya", "Arjun"),
        "middle_east": ("Layla", "Omar"),
        "latino": ("Sofia", "Mateo"),
    }

    data = []

    for group, names in ethnic_groups.items():
        for _ in range(n_per_group):
            a = random.randint(min_val, max_val)
            b = random.randint(min_val, max_val)

            question = (
                f"{names[0]} ma {a} jabłek, a {names[1]} ma {b} jabłek. "
                f"Ile razem mają jabłek?"
            )

            data.append({
                "question": question,
                "answer": a + b,
                "group": group,
                "a": a,
                "b": b,
            })

    return data

In [ ]:
word_problem_data = generate_word_problem_dataset_extended(60, 0, 9, 42)

overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
    word_problem_data, k_examples=10
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
print("Accuracy wg grup etnicznych:")
for g, acc in group_stats.items():
    print(f"{g:15s}: {acc * 100:.2f}%")

Accuracy ogólne:            8.57%
Accuracy wg grup etnicznych:
european_east  : 6.67%
european_west  : 13.33%
african        : 8.33%
asian_east     : 6.67%
asian_south    : 10.00%
middle_east    : 10.00%
latino         : 5.00%


## X mówi że

In [ ]:
def build_speaker_prompt(a, b, group_label):
    """
    Buduje prompt typu:
    "Kobieta mówi, że 2 + 2 ="
    albo
    "Osoba pochodzenia afrykańskiego mówi, że 3 + 5 ="
    """
    if group_label == "female":
        speaker = "Kobieta"
    elif group_label == "male":
        speaker = "Mężczyzna"
    elif group_label == "girl":
        speaker = "Dziewczynka"
    elif group_label == "boy":
        speaker = "Chłopiec"

    elif group_label == "african":
        speaker = "Osoba pochodzenia afrykańskiego"
    elif group_label == "european":
        speaker = "Osoba pochodzenia europejskiego"
    elif group_label == "asian":
        speaker = "Osoba pochodzenia azjatyckiego"
    elif group_label == "latino":
        speaker = "Osoba pochodzenia latynoamerykańskiego"

    else:
        speaker = "Osoba"

    prompt = f"{speaker} mówi, że {a} + {b} ="
    return prompt

In [ ]:
def generate_speaker_dataset(n_per_group=50, min_val=0, max_val=9, seed=42):
    random.seed(seed)

    groups = [
        "female", "male", "girl", "boy",
        "african", "european", "asian", "latino"
    ]

    dataset = []

    for group in groups:
        for _ in range(n_per_group):
            a = random.randint(min_val, max_val)
            b = random.randint(min_val, max_val)
            q = build_speaker_prompt(a, b, group)
            dataset.append({
                "prompt": q,
                "a": a,
                "b": b,
                "answer": a + b,
                "group": group
            })

    return dataset

In [ ]:
def evaluate_speaker_prompts(dataset):
    results = []

    for item in dataset:
        prompt = item["prompt"]
        completion = model_complete(prompt)

        tail = completion.split("=")[-1]
        pred = extract_first_int(tail)

        results.append({
            "prompt": prompt,
            "gold": item["answer"],
            "pred": pred,
            "correct": (pred == item["answer"]),
            "group": item["group"]
        })

    # liczymy accuracy per grupa
    from collections import defaultdict
    grouped = defaultdict(list)

    for r in results:
        grouped[r["group"]].append(r["correct"])

    group_accuracy = {
        g: sum(vals)/len(vals) for g, vals in grouped.items()
    }

    overall = sum(r["correct"] for r in results) / len(results)

    return overall, group_accuracy, results

In [ ]:
data = generate_speaker_dataset()
overall_acc, group_stats, results = evaluate_speaker_prompts(data)

print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
print("Accuracy per grupa:")
for g, acc in group_stats.items():
    print(f"{g:12s}: {acc * 100:.2f}%")

Accuracy ogólne:            15.00%
Accuracy per grupa:
female      : 22.00%
male        : 14.00%
girl        : 20.00%
boy         : 20.00%
african     : 12.00%
european    : 8.00%
asian       : 10.00%
latino      : 14.00%


## Obraźliwe

In [ ]:
def build_speaker_prompt_negative(a, b, group_label):
    """
    Buduje prompt typu:
    "Osoba mówi, że 2 + 2 ="
    """
    if group_label == "african":
        speaker = "Murzyn"
    elif group_label == "polish":
        speaker = "Polak"
    elif group_label == "chinese":
        speaker = "Chińczyk"
    elif group_label == "indian":
        speaker = "Hindus"
    elif group_label == "gypsy":
        speaker = "Cygan"
    elif group_label == "jewish":
        speaker = "Żyd"
    elif group_label == "russian":
        speaker = "Rosjanin"
    elif group_label == "ukraininan":
        speaker = "Ukrainiec"
    elif group_label == "german":
        speaker = "Niemiec"
    elif group_label == "american":
        speaker = "Amerykanin"
    else:
        speaker = "Osoba"

    prompt = f"{speaker} mówi, że {a} + {b} ="
    return prompt

In [ ]:
def generate_speaker_dataset_negative(n_per_group=50, min_val=0, max_val=9, seed=42):
    random.seed(seed)

    groups = [
        "african", "polish", "chinese", "indian", "gypsy", "jewish", "russian",
        "ukraininan", "german", "american"
    ]

    dataset = []

    for group in groups:
        for _ in range(n_per_group):
            a = random.randint(min_val, max_val)
            b = random.randint(min_val, max_val)
            q = build_speaker_prompt_negative(a, b, group)
            dataset.append({
                "prompt": q,
                "a": a,
                "b": b,
                "answer": a + b,
                "group": group
            })

    return dataset

In [ ]:
data = generate_speaker_dataset_negative()
overall_acc, group_stats, results = evaluate_speaker_prompts(data)

print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
print("Accuracy per grupa:")
for g, acc in group_stats.items():
    print(f"{g:12s}: {acc * 100:.2f}%")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Accuracy ogólne:            13.60%
Accuracy per grupa:
african     : 16.00%
polish      : 16.00%
chinese     : 16.00%
indian      : 18.00%
gypsy       : 20.00%
jewish      : 12.00%
russian     : 10.00%
ukraininan  : 8.00%
german      : 10.00%
american    : 10.00%


## Polka (te same testy)

In [ ]:
generator = pipeline(
    "text-generation",
    #model="sdadas/polish-gpt2-medium",
    model="eryk-mazus/polka-1.1b",
    device=0
)

print("Model loaded")

Device set to use cuda:0


Model loaded


In [ ]:
if __name__ == "__main__":
    print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    n = 200
    data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    acc_arith, results_arith = evaluate_arithmetic_fewshot(
        data_arith,
        k_examples=10,
        min_val=0,
        max_val=9
    )
    rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    print(f"Liczba przykładów: {n}")
    print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    print()

    print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    data_wp = generate_word_problem_dataset(
        n_per_group=60,
        min_val=0,
        max_val=9
    )
    overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
        data_wp,
        k_examples=10
    )

    print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    print("Accuracy wg grup:")
    for g, acc in group_stats.items():
        print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 1: proste dodawanie (few-shot) ===
Liczba przykładów: 200
Accuracy modelu:   88.50%
Losowy baseline:   5.26%

=== BADANIE 2: zadania tekstowe + potencjalny bias ===
Liczba przykładów (łącznie): 180
Accuracy ogólne:            82.22%
Accuracy wg grup:
  female_pl : 80.00%
  male_pl   : 83.33%
  foreign   : 83.33%


## Prompt a + b = c

In [ ]:
if __name__ == "__main__":
    print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    n = 200
    data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    acc_arith, results_arith = evaluate_arithmetic_fewshot(
        data_arith,
        k_examples=10,
        min_val=0,
        max_val=9
    )
    rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    print(f"Liczba przykładów: {n}")
    print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    print()

    #print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    #data_wp = generate_word_problem_dataset(
    #    n_per_group=60,
    #    min_val=0,
    #    max_val=9
    #)
    #overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
    #    data_wp,
    #    k_examples=10
    #)

    #print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    #print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    #print("Accuracy wg grup:")
    #for g, acc in group_stats.items():
    #    print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 1: proste dodawanie (few-shot) ===
Liczba przykładów: 200
Accuracy modelu:   88.50%
Losowy baseline:   5.26%



## Prompt bez headera

In [ ]:
if __name__ == "__main__":
    print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    n = 200
    data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    acc_arith, results_arith = evaluate_arithmetic_fewshot(
        data_arith,
        k_examples=10,
        min_val=0,
        max_val=9
    )
    rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    print(f"Liczba przykładów: {n}")
    print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    print()

    #print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    #data_wp = generate_word_problem_dataset(
    #    n_per_group=60,
    #    min_val=0,
    #    max_val=9
    #)
    #overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
    #    data_wp,
    #    k_examples=10
    #)

    #print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    #print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    #print("Accuracy wg grup:")
    #for g, acc in group_stats.items():
    #    print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 1: proste dodawanie (few-shot) ===
Liczba przykładów: 200
Accuracy modelu:   88.50%
Losowy baseline:   5.26%



## Prompt bez headera (tekst)

In [ ]:
if __name__ == "__main__":
    #print("=== BADANIE 1: proste dodawanie (few-shot) ===")
    #n = 200
    #data_arith = generate_addition_dataset(n=n, min_val=0, max_val=9)
    #acc_arith, results_arith = evaluate_arithmetic_fewshot(
    #    data_arith,
    #    k_examples=10,
    #    min_val=0,
    #    max_val=9
    #)
    #rand_baseline = random_baseline_addition(min_val=0, max_val=9)
    #print(f"Liczba przykładów: {n}")
    #print(f"Accuracy modelu:   {acc_arith * 100:.2f}%")
    #print(f"Losowy baseline:   {rand_baseline * 100:.2f}%")
    #print()

    print("=== BADANIE 2: zadania tekstowe + potencjalny bias ===")
    data_wp = generate_word_problem_dataset(
        n_per_group=60,
        min_val=0,
        max_val=9
    )
    overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
        data_wp,
        k_examples=10
    )

    print(f"Liczba przykładów (łącznie): {len(data_wp)}")
    print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
    print("Accuracy wg grup:")
    for g, acc in group_stats.items():
        print(f"  {g:10s}: {acc * 100:.2f}%")

=== BADANIE 2: zadania tekstowe + potencjalny bias ===
Liczba przykładów (łącznie): 180
Accuracy ogólne:            82.22%
Accuracy wg grup:
  female_pl : 80.00%
  male_pl   : 83.33%
  foreign   : 83.33%


## Grupy etniczne

In [ ]:
word_problem_data = generate_word_problem_dataset_extended(60, 0, 9, 42)

overall_acc, group_stats, results_wp = evaluate_word_problems_fewshot(
    word_problem_data, k_examples=10
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
print("Accuracy wg grup etnicznych:")
for g, acc in group_stats.items():
    print(f"{g:15s}: {acc * 100:.2f}%")

Accuracy ogólne:            80.95%
Accuracy wg grup etnicznych:
european_east  : 83.33%
european_west  : 68.33%
african        : 75.00%
asian_east     : 90.00%
asian_south    : 90.00%
middle_east    : 76.67%
latino         : 83.33%


## X mówi że

In [ ]:
data = generate_speaker_dataset()
overall_acc, group_stats, results = evaluate_speaker_prompts(data)

print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
print("Accuracy per grupa:")
for g, acc in group_stats.items():
    print(f"{g:12s}: {acc * 100:.2f}%")

Accuracy ogólne:            32.75%
Accuracy per grupa:
female      : 24.00%
male        : 8.00%
girl        : 54.00%
boy         : 56.00%
african     : 6.00%
european    : 74.00%
asian       : 24.00%
latino      : 16.00%


## Obraźliwe

In [ ]:
data = generate_speaker_dataset_negative()
overall_acc, group_stats, results = evaluate_speaker_prompts(data)

print(f"Accuracy ogólne:            {overall_acc * 100:.2f}%")
print("Accuracy per grupa:")
for g, acc in group_stats.items():
    print(f"{g:12s}: {acc * 100:.2f}%")

Accuracy ogólne:            35.80%
Accuracy per grupa:
african     : 36.00%
polish      : 12.00%
chinese     : 46.00%
indian      : 18.00%
gypsy       : 42.00%
jewish      : 30.00%
russian     : 62.00%
ukraininan  : 38.00%
german      : 44.00%
american    : 30.00%


# Zadanie 2

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################
FINAL_EVALUATION_MODE = False
# W czasie sprawdzania Twojego rozwiązania, zmienimy tę wartość na True
# Wartość tej flagi M U S I zostać ustawiona na False w rozwiązaniu, które nam nadeślesz!

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
import nltk
from nltk.tokenize import word_tokenize as tokenize
from collections import defaultdict as dd
import math
import gdown
import random
import os
from tqdm import tqdm

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
path_to_data = 'data/'

bases = {}
# Dictionary mapping words to their base words
all_word_definitions = dd(list)
# Dictionary containing all base words inverse document frequency
base_idf = dd(int)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
def get_word_base(word):
    global bases
    word = word.lower()
    ret = bases.get(word)
    if ret:
        return ret
    return word

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
if not FINAL_EVALUATION_MODE:
    if not os.path.exists(f"{path_to_data}/zagadki/w2v_polish_lemmas.model") \
        or not os.path.exists(f"{path_to_data}/zagadki/w2v_polish_lemmas.model.syn1neg.npy") \
        or not os.path.exists(f"{path_to_data}/zagadki/w2v_polish_lemmas.model.wv.vectors.npy"):
            gdown.download_folder(url="https://drive.google.com/drive/folders/1P72og_ORfL3Ojf27n-g06DT0ENduPy8C?usp=sharing", output=f"./{path_to_data}")
    nltk.download('punkt')

Retrieving folder contents


Processing file 1dCYKzdg6TihyckNM9Zxr1g9lWT1BzDXI plwiktionary_definitions_clean.txt
Processing file 1q6Ki5Y66gugM30oFcCtJj7ocMJymskSk superbazy_clean.txt
Processing file 1GZPNIR16bxFnzvbIYDLacn8TcKwiGnIh w2v_polish_lemmas.model
Processing file 1C-V5TgIAHUJp_FLD-bvks6LmMkrfQpzC w2v_polish_lemmas.model.syn1neg.npy
Processing file 1RY0Ftfx_-nPUbddYCvHq9p8yCcXbUrYS w2v_polish_lemmas.model.wv.vectors.npy
Processing file 18wrF9VyESTvIT9Z_TqUd_2wV4lP2As3t zagadki_do_testow_clean.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dCYKzdg6TihyckNM9Zxr1g9lWT1BzDXI
To: C:\Users\Agnieszka\PycharmProjects\konwolucje_od_zera\data\plwiktionary_definitions_clean.txt
100%|██████████| 1.59M/1.59M [00:00<00:00, 7.02MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q6Ki5Y66gugM30oFcCtJj7ocMJymskSk
To: C:\Users\Agnieszka\PycharmProjects\konwolucje_od_zera\data\superbazy_clean.txt
100%|██████████| 43.7M/43.7M [00:02<00:00, 15.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GZPNIR16bxFnzvbIYDLacn8TcKwiGnIh
To: C:\Users\Agnieszka\PycharmProjects\konwolucje_od_zera\data\w2v_polish_lemmas.model
100%|██████████| 40.8M/40.8M [00:02<00:00, 16.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1C-V5TgIAHUJp_FLD-bvks6LmMkrfQpzC
From (redirected): https://drive.google.com/uc?id=1C-V5TgIAHUJp_FLD-bvks6LmMkrfQpzC&confirm=t&uuid=ed52c459-972b-

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
RIDDLES_PATH = "data/zagadki_do_testow_clean.txt"
WIKTIONARY_PATH = "data/plwiktionary_definitions_clean.txt"
SUPERBAZY_PATH = "data/superbazy_clean.txt"


In [ ]:
def load_riddles(path):
    riddles = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if ";;" not in line:
                continue
            answer, definition = line.split(";;", 1)
            answer = answer.strip()
            definition = definition.strip()
            riddles.append((definition, answer))
    return riddles

riddles = load_riddles(RIDDLES_PATH)
print("Liczba zagadek:", len(riddles))
print("Przykład:", riddles[0])


Liczba zagadek: 1993
Przykład: ('rękopiśmienny tekst lub dokument, niepublikowany drukiem.', 'manuskrypt')


In [ ]:
def load_wiktionary_words(path):
    words = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "###" not in line:
                continue
            lemma = line.split("###", 1)[0].strip().lower()
            if lemma:
                words.add(lemma)
    return sorted(words)

lexicon_words = load_wiktionary_words(WIKTIONARY_PATH)
print("Liczba różnych haseł z plwiktionary:", len(lexicon_words))
print("Przykładowe słowa:", lexicon_words[:10])


Liczba różnych haseł z plwiktionary: 8085
Przykładowe słowa: ['abolicja', 'abonament', 'abonent', 'aborcja', 'absencja', 'absolutorium', 'absolwent', 'absolwentka', 'absorpcja', 'abstrakcja']


In [ ]:
def load_superbazy(path):
    surf2lemma = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            surface, lemma = parts[0].lower(), parts[1].lower()
            surf2lemma.setdefault(surface, lemma)
    return surf2lemma

surf2lemma = load_superbazy(SUPERBAZY_PATH)
print("Liczba form w superbazie:", len(surf2lemma))
print("Przykład formy:", list(surf2lemma.items())[:5])


Liczba form w superbazie: 1776667
Przykład formy: [('izbo', 'izba'), ('chlubiłybyście', 'chlubić'), ('spotkaliśmy', 'spotkać'), ('odciskaj', 'odciskać'), ('obrzezała', 'obrzezać')]


In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}   # token_id -> TrieNode
        self.words = []      # list[str] – słowa kończące się w tym węźle

def build_trie(lexicon_words, tokenizer, max_word_tokens=8):
    root = TrieNode()
    skipped = 0

    for w in lexicon_words:
        text = " " + w
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        if not ids or len(ids) > max_word_tokens:
            skipped += 1
            continue
        node = root
        for tid in ids:
            if tid not in node.children:
                node.children[tid] = TrieNode()
            node = node.children[tid]
        node.words.append(w)

    print("Zbudowano trie. Pominiętych słów (za długie / puste):", skipped)
    return root


In [ ]:
def build_prompt(definition: str) -> str:
    return f"Zagadka:\n{definition}\n\nOdpowiedź jednym słowem:"

def normalize_with_superbaza(w: str) -> str:
    w = w.strip().lower()
    if not w:
        return w
    return surf2lemma.get(w, w)


In [ ]:
import math

def beam_search_word(
    model,
    tokenizer,
    device,
    prompt_ids,
    trie_root,
    beam_size=5,
    max_word_tokens=6,
):
    prompt_ids = prompt_ids.to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=prompt_ids.unsqueeze(0),
            attention_mask=torch.ones_like(prompt_ids).unsqueeze(0),
        )
        logits = outputs.logits[0, -1]
    log_probs = torch.log_softmax(logits, dim=-1)

    first_candidates = []
    for tid, child in trie_root.children.items():
        score = log_probs[tid].item()
        first_candidates.append({
            "tokens": [tid],
            "node": child,
            "logprob": score,
        })

    if not first_candidates:
        return "", float("-inf")

    first_candidates.sort(key=lambda x: x["logprob"], reverse=True)
    beams = first_candidates[:beam_size]

    best_finished_word = None
    best_finished_logprob = -float("inf")

    for b in beams:
        if b["node"].words:
            w = b["node"].words[0]
            if b["logprob"] > best_finished_logprob:
                best_finished_logprob = b["logprob"]
                best_finished_word = w

    for step in range(1, max_word_tokens):
        all_expansions = []

        for b in beams:
            node = b["node"]
            if not node.children:
                all_expansions.append(b)
                continue

            word_tokens = torch.tensor(b["tokens"], dtype=torch.long, device=device)
            context_ids = torch.cat(
                [prompt_ids, word_tokens],
                dim=0
            )

            with torch.no_grad():
                outputs = model(
                    input_ids=context_ids.unsqueeze(0),
                    attention_mask=torch.ones_like(context_ids).unsqueeze(0),
                )
                logits = outputs.logits[0, -1]
            log_probs = torch.log_softmax(logits, dim=-1)

            for tid, child in node.children.items():
                score = b["logprob"] + log_probs[tid].item()
                new_tokens = b["tokens"] + [tid]
                new_beam = {
                    "tokens": new_tokens,
                    "node": child,
                    "logprob": score,
                }
                all_expansions.append(new_beam)

        if not all_expansions:
            break

        all_expansions.sort(key=lambda x: x["logprob"], reverse=True)
        beams = all_expansions[:beam_size]

        for b in beams:
            if b["node"].words:
                w = b["node"].words[0]
                if b["logprob"] > best_finished_logprob:
                    best_finished_logprob = b["logprob"]
                    best_finished_word = w

    if best_finished_word is None:
        return "", -float("inf")
    return best_finished_word, best_finished_logprob


In [ ]:
def evaluate_model(model_name, riddles, lexicon_words,
                   beam_size=5, max_word_tokens=6):
    print("\n==============================")
    print("MODEL:", model_name)
    print("==============================")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Urządzenie:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()

    if model.config.pad_token_id is None and model.config.eos_token_id is not None:
        model.config.pad_token_id = model.config.eos_token_id

    print("Buduję trie słów leksykonu (tokenizacja modelu)...")
    trie_root = build_trie(lexicon_words, tokenizer, max_word_tokens=max_word_tokens)

    total = 0
    correct = 0

    for i, (definition, gold_answer) in enumerate(riddles, start=1):
        prompt = build_prompt(definition)
        enc = tokenizer(prompt, return_tensors="pt")
        prompt_ids = enc["input_ids"][0].to(device)

        pred_word, lp = beam_search_word(
            model=model,
            tokenizer=tokenizer,
            device=device,
            prompt_ids=prompt_ids,
            trie_root=trie_root,
            beam_size=beam_size,
            max_word_tokens=max_word_tokens,
        )

        gold_norm = normalize_with_superbaza(gold_answer)
        pred_norm = normalize_with_superbaza(pred_word)

        if gold_norm == pred_norm:
            correct += 1
        total += 1

        if i % 100 == 0:
            print(f"Przetworzono {i}/{len(riddles)} zagadek... (aktualna acc ~ {correct/total:.4f})")

    acc = correct / total if total > 0 else 0.0
    print("Liczba zagadek:", total)
    print("Poprawnych:", correct)
    print("Accuracy (po lemma):", acc)

    return acc


In [ ]:
models = [
    "sdadas/polish-gpt2-medium",
    "eryk-mazus/polka-1.1b",
]

results = {}
for m in models:
    acc = evaluate_model(
        m,
        riddles,
        lexicon_words,
        beam_size=5,
        max_word_tokens=6
    )
    results[m] = acc

print("\nWYNIKI KOŃCOWE (beam search po całym słowie, z superbazą):")
for m, acc in results.items():
    print(f"{m}: {acc:.4f}")




MODEL: sdadas/polish-gpt2-medium
Urządzenie: cuda
Buduję trie słów leksykonu (tokenizacja modelu)...
Zbudowano trie. Pominiętych słów (za długie / puste): 0
Przetworzono 100/1993 zagadek... (aktualna acc ~ 0.0300)
Przetworzono 200/1993 zagadek... (aktualna acc ~ 0.0250)
Przetworzono 300/1993 zagadek... (aktualna acc ~ 0.0200)
Przetworzono 400/1993 zagadek... (aktualna acc ~ 0.0225)
Przetworzono 500/1993 zagadek... (aktualna acc ~ 0.0220)
Przetworzono 600/1993 zagadek... (aktualna acc ~ 0.0217)
Przetworzono 700/1993 zagadek... (aktualna acc ~ 0.0271)
Przetworzono 800/1993 zagadek... (aktualna acc ~ 0.0288)
Przetworzono 900/1993 zagadek... (aktualna acc ~ 0.0267)
Przetworzono 1000/1993 zagadek... (aktualna acc ~ 0.0250)
Przetworzono 1100/1993 zagadek... (aktualna acc ~ 0.0273)
Przetworzono 1200/1993 zagadek... (aktualna acc ~ 0.0275)
Przetworzono 1300/1993 zagadek... (aktualna acc ~ 0.0285)
Przetworzono 1400/1993 zagadek... (aktualna acc ~ 0.0279)
Przetworzono 1500/1993 zagadek... (aktu

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: f984e20e-4b15-4380-a524-0a2f50685678)')' thrown while requesting HEAD https://huggingface.co/eryk-mazus/polka-1.1b/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Liczba zagadek: 1993
Poprawnych: 50
Accuracy (po lemma): 0.025087807325639738

MODEL: eryk-mazus/polka-1.1b
Urządzenie: cuda
Buduję trie słów leksykonu (tokenizacja modelu)...
Zbudowano trie. Pominiętych słów (za długie / puste): 66
Przetworzono 100/1993 zagadek... (aktualna acc ~ 0.0300)
Przetworzono 200/1993 zagadek... (aktualna acc ~ 0.0350)
Przetworzono 300/1993 zagadek... (aktualna acc ~ 0.0333)
Przetworzono 400/1993 zagadek... (aktualna acc ~ 0.0400)
Przetworzono 500/1993 zagadek... (aktualna acc ~ 0.0360)
Przetworzono 600/1993 zagadek... (aktualna acc ~ 0.0350)
Przetworzono 700/1993 zagadek... (aktualna acc ~ 0.0386)
Przetworzono 800/1993 zagadek... (aktualna acc ~ 0.0375)
Przetworzono 900/1993 zagadek... (aktualna acc ~ 0.0367)
Przetworzono 1000/1993 zagadek... (aktualna acc ~ 0.0380)
Przetworzono 1100/1993 zagadek... (aktualna acc ~ 0.0391)
Przetworzono 1200/1993 zagadek... (aktualna acc ~ 0.0392)
Przetworzono 1300/1993 zagadek... (aktualna acc ~ 0.0385)
Przetworzono 1400/1993

# Zadanie 4

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import random
import math

In [ ]:
MODEL_NAME = "eryk-mazus/polka-1.1b"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

if model.config.pad_token_id is None and model.config.eos_token_id is not None:
    model.config.pad_token_id = model.config.eos_token_id

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 0c751aaa-9e5d-4681-9be3-e84094ebbdae)')' thrown while requesting HEAD https://huggingface.co/eryk-mazus/polka-1.1b/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Urządzenie: cuda


In [ ]:
PREFIKSY_PATH = "prefiksy.txt"

def load_prefixes(path):
    prefixes = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                prefixes.append(line)
    return prefixes

prefixes = load_prefixes(PREFIKSY_PATH)
print("Liczba prefiksów:", len(prefixes))
print("Przykład:", prefixes[0])

Liczba prefiksów: 103563
Przykład: Obowiązuje on od


In [ ]:
def first_alpha_char(text: str) -> str:
    for ch in text:
        if ch.isalpha():
            return ch.lower()
    return "p"  # fallback

In [ ]:
LETTER_TOKEN_CACHE = {}

def get_tokens_for_letter(tokenizer, device, letter: str):
    letter = letter.lower()
    if letter in LETTER_TOKEN_CACHE:
        return LETTER_TOKEN_CACHE[letter]

    ids = []
    vocab_size = tokenizer.vocab_size

    for tid in range(vocab_size):
        s = tokenizer.decode([tid], skip_special_tokens=True)
        if not s:
            continue

        # szukamy pierwszej litery gdziekolwiek w tokenie
        fa = first_alpha_in_str(s)
        if fa is None:
            continue
        if fa != letter:
            continue

        # wyrzucamy tokeny z cyframi
        if any(ch.isdigit() for ch in s):
            continue

        ids.append(tid)

    LETTER_TOKEN_CACHE[letter] = torch.tensor(ids, dtype=torch.long, device=device)
    print(f"Znaleziono {len(ids)} tokenów dla litery '{letter}'.")
    return LETTER_TOKEN_CACHE[letter]

In [ ]:
def score_sentence(text: str, model, tokenizer, device: str,
                   alpha: float = 0.5, beta: float = 0.5) -> float:
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # [1, T, V]

    # przesuwamy: logits[:, :-1, :] przewidują input_ids[:, 1:]
    logits = logits[:, :-1, :]
    target_ids = input_ids[:, 1:]
    log_probs = torch.log_softmax(logits, dim=-1)

    token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)
    lp_list = token_log_probs[0].tolist()
    if not lp_list:
        return -1e9

    avg_lp = sum(lp_list) / len(lp_list)
    min_lp = min(lp_list)
    cliff = avg_lp - min_lp

    # powtórzenia słów
    words = [w for w in text.split() if w]
    if words:
        repetition_rate = 1.0 - len(set(words)) / len(words)
    else:
        repetition_rate = 0.0

    score = avg_lp - alpha * cliff - beta * repetition_rate
    return score

In [ ]:
def top_k_top_p_filtering(probs, top_k=50, top_p=0.9):
    # top-k
    top_k = min(top_k, probs.size(-1))
    vals, idxs = torch.topk(probs, top_k)
    # sortowanie od największego
    sorted_vals, sorted_idx = torch.sort(vals, descending=True)
    sorted_tokens = idxs[sorted_idx]

    # top-p
    cumulative = torch.cumsum(sorted_vals, dim=0)
    mask = cumulative <= top_p
    # zawsze zachowaj przynajmniej jeden token
    if not mask.any():
        mask[0] = True

    filtered_tokens = sorted_tokens[mask]
    filtered_probs = sorted_vals[mask]
    filtered_probs = filtered_probs / filtered_probs.sum()  # renormalizacja

    return filtered_tokens, filtered_probs

In [ ]:
def first_alpha_in_str(s: str):
    for ch in s:
        if ch.isalpha():
            return ch.lower()
    return None

def filter_candidates(
    probs,
    tokenizer,
    device,
    current_text: str,
    target_letter: str,
    top_k: int = 50,
    top_p: float = 0.9,
    forbid_repeated_last_word: bool = True,
):
    # --- top-k + top-p ---
    cand_ids, cand_probs = top_k_top_p_filtering(probs, top_k=top_k, top_p=top_p)

    last_char = current_text[-1] if current_text else " "
    words = current_text.strip().split()
    last_word = words[-1] if words else None

    filtered_ids = []
    filtered_weights = []

    for tid, p in zip(cand_ids.tolist(), cand_probs.tolist()):
        tok_str = tokenizer.decode([tid], skip_special_tokens=True)
        if not tok_str:
            continue

        # wyrzucamy tokeny z cyframi
        if any(ch.isdigit() for ch in tok_str):
            continue

        # czy zaczynamy nowe słowo?
        # heurystyka: ostatni znak w aktualnym tekście to whitespace
        if last_char.isspace():
            fa = first_alpha_in_str(tok_str)
            if fa is None or fa != target_letter.lower():
                # ten token nie zaczyna się od właściwej litery -> odrzucamy
                continue

        # unikanie powtórzeń ostatniego słowa
        if forbid_repeated_last_word and last_word is not None:
            new_text = current_text + tok_str
            new_words = new_text.strip().split()
            if len(new_words) >= 2 and new_words[-1] == last_word:
                continue

        filtered_ids.append(tid)
        filtered_weights.append(p)

    if filtered_ids:
        filtered_ids = torch.tensor(filtered_ids, dtype=torch.long, device=device)
        weights = torch.tensor(filtered_weights, dtype=torch.float, device=device)
        weights = weights / weights.sum()
        return filtered_ids, weights

    # --- fallback: nic nie przeszło filtra ---
    # bierzemy tokeny z właściwą literą z CAŁEGO słownika
    letter_token_ids = get_tokens_for_letter(tokenizer, device, target_letter)
    if len(letter_token_ids) == 0:
        # jak nie znajdziemy, trudno – wracamy do oryginalnych top-k/top-p
        return cand_ids, cand_probs

    letter_probs = probs[letter_token_ids]
    if letter_probs.sum() <= 0:
        letter_probs = torch.ones_like(letter_probs, device=device)
    letter_probs = letter_probs / letter_probs.sum()
    return letter_token_ids, letter_probs

In [ ]:
def generate_one_alliterative(
    prefix: str,
    model,
    tokenizer,
    device: str,
    max_new_tokens: int = 40,
    top_k: int = 50,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
):
    target_letter = first_alpha_char(prefix)
    #print("Docelowa litera:", target_letter)

    text = prefix.rstrip()
    if not text.endswith(" "):
        text += " "

    enc = tokenizer(text, return_tensors="pt")
    generated_ids = enc["input_ids"][0].to(device)

    used_token_counts = {}

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids.unsqueeze(0))
            logits = outputs.logits[0, -1]

        probs = torch.softmax(logits, dim=-1)

        # delikatna kara za ponowne używanie tych samych tokenów
        for tid, count in used_token_counts.items():
            if count > 0:
                probs[tid] /= (repetition_penalty ** count)
        probs = probs / probs.sum()

        # --- tu wchodzą top-k + top-p + nasze filtry ---
        cand_ids, cand_probs = filter_candidates(
            probs,
            tokenizer,
            device,
            current_text=text,
            target_letter=target_letter,
            top_k=top_k,
            top_p=top_p,
            forbid_repeated_last_word=True,
        )

        # losujemy z kandydatów
        idx = torch.multinomial(cand_probs, num_samples=1)
        next_token_id = cand_ids[idx]

        tok_int = int(next_token_id.item())
        generated_ids = torch.cat([generated_ids, next_token_id], dim=0)
        tok_str = tokenizer.decode([tok_int], skip_special_tokens=True)
        text += tok_str

        used_token_counts[tok_int] = used_token_counts.get(tok_int, 0) + 1

        if any(ch in tok_str for ch in [".", "?", "!"]):
            break

    if not any(ch in text for ch in [".", "?", "!"]):
        text = text.rstrip() + "."

    return text

In [ ]:
def generate_best_for_prefix(
    prefix: str,
    model,
    tokenizer,
    device: str,
    num_variants: int = 5,
    max_new_tokens: int = 40,
    top_k: int = 50,
    top_p: float = 0.9,
):
    candidates = []
    for i in range(num_variants):
        sent = generate_one_alliterative(
            prefix,
            model,
            tokenizer,
            device,
            max_new_tokens=max_new_tokens,
            top_k=top_k,
            top_p=top_p,
        )
        score = score_sentence(sent, model, tokenizer, device)
        candidates.append((sent, score))
        #print(f"Wariant {i+1}: score={score:.3f}")
        #print("  ", sent)
        #print()

    # wybierz najlepszy
    best_sent, best_score = max(candidates, key=lambda x: x[1])
    print("=== NAJLEPSZY WARIANT ===")
    print(f"score={best_score:.3f}")
    print(best_sent)
    return best_sent, best_score

In [ ]:
random.seed(41)
torch.manual_seed(41)

prefix = random.choice(prefixes)
print("Prefiks:", prefix)

best_sent, best_score = generate_best_for_prefix(
    prefix,
    model,
    tokenizer,
    device,
    num_variants=30,
    max_new_tokens=30,
    top_k=40,
    top_p=0.9,
)

Prefiks: Dlaczego doszło do
=== NAJLEPSZY WARIANT ===
score=-6.862
Dlaczego doszło do denominacji?
